# 🔲 Redução de Dimensionalidade — Binarização de Imagem

**Desafio DIO · Visão Computacional com OpenCV**

Objetivo: converter uma imagem colorida em:
1. **Escala de cinza** (valores 0 a 255)
2. **Imagem binária** — preto e branco puro (0 ou 255)

---

## 📦 1. Instalação de dependências
*(executar apenas no Google Colab)*

In [ ]:
# Verifica e instala opencv se necessário
try:
    import cv2
    print(f"OpenCV já instalado: v{cv2.__version__}")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "opencv-python-headless", "-q"])
    import cv2
    print(f"OpenCV instalado: v{cv2.__version__}")

## 📚 2. Importações

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print("Bibliotecas carregadas com sucesso!")
print(f"  OpenCV  : {cv2.__version__}")
print(f"  NumPy   : {np.__version__}")

## 📂 3. Carregar imagem

**Formatos aceitos:** `.jpg`, `.jpeg`, `.png`, `.bmp`, `.tiff`

> **Colab:** execute a célula abaixo para fazer upload direto do seu computador.  
> **VS Code / local:** ajuste a variável `CAMINHO_IMAGEM` para o caminho da sua imagem.

In [ ]:
# ── Escolha UMA das opções abaixo ────────────────────────────────────────

# Opção A: Upload no Google Colab
try:
    from google.colab import files
    print("Ambiente: Google Colab")
    uploaded = files.upload()
    CAMINHO_IMAGEM = list(uploaded.keys())[0]

# Opção B: Caminho local (VS Code ou Jupyter local)
except ImportError:
    print("Ambiente: local (VS Code / Jupyter)")
    CAMINHO_IMAGEM = "assets/lena.jpg"   # <-- altere para o caminho da sua imagem

print(f"\nImagem selecionada: {CAMINHO_IMAGEM}")

## 🖼️ 4. Processamento — Cinza e Binarização

In [ ]:
# ── Parâmetro principal ───────────────────────────────────────────────────
LIMIAR = 127   # 0–255: ajuste conforme necessário

# ── Passo 1: Carregar imagem colorida ────────────────────────────────────
img_bgr = cv2.imread(CAMINHO_IMAGEM)
if img_bgr is None:
    raise FileNotFoundError(f"Não foi possível abrir '{CAMINHO_IMAGEM}'. Verifique o caminho.")

# OpenCV carrega em BGR; converter para RGB para exibir corretamente no Matplotlib
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# ── Passo 2: Converter para escala de cinza ───────────────────────────────
# Reduz 3 canais (RGB) para 1 canal — valores 0 a 255
img_cinza = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# ── Passo 3: Binarizar ────────────────────────────────────────────────────
# Pixels > LIMIAR → 255 (branco) | Pixels <= LIMIAR → 0 (preto)
_, img_bin = cv2.threshold(img_cinza, LIMIAR, 255, cv2.THRESH_BINARY)

# ── Informações ───────────────────────────────────────────────────────────
h, w = img_bgr.shape[:2]
print(f"Imagem carregada: {w} x {h} px")
print(f"Original  → shape: {img_rgb.shape}    | canais: {img_bgr.shape[2]}")
print(f"Cinza     → shape: {img_cinza.shape} | dtype: {img_cinza.dtype}")
print(f"Binária   → valores únicos: {np.unique(img_bin)}")

## 📊 5. Visualização dos Resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

dados = [
    (img_rgb,    "Imagem Original",  None),
    (img_cinza,  "Imagem em Cinza",  "gray"),
    (img_bin,    "Imagem Binária",   "gray"),
]

for ax, (img, titulo, cmap) in zip(axes, dados):
    ax.imshow(img, cmap=cmap)
    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=10)
    ax.axis("off")

plt.suptitle("Redução de Dimensionalidade — Binarização", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("resultado_binarizacao.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: resultado_binarizacao.png")

## 🔬 6. Versão Avançada — com Blur Gaussiano

Reproduz o exemplo do slide da aula: suaviza a imagem antes de binarizar,
gerando também a versão invertida e uma máscara `bitwise_and`.

In [ ]:
LIMIAR_AV = 160   # limiar usado no slide da aula

# Blur Gaussiano: reduz ruídos antes do threshold
suave = cv2.GaussianBlur(img_cinza, (7, 7), 0)

# Binarizações
_, bin_normal = cv2.threshold(suave, LIMIAR_AV, 255, cv2.THRESH_BINARY)
_, bin_inv    = cv2.threshold(suave, LIMIAR_AV, 255, cv2.THRESH_BINARY_INV)

# Máscara: aplica binário invertido sobre a imagem cinza (igual ao slide)
mascara = cv2.bitwise_and(img_cinza, img_cinza, mask=bin_inv)

# Painel 2x2
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
pares = [
    (suave,      "Cinza + Blur Gaussiano"),
    (bin_normal, "Binarizada Normal"),
    (bin_inv,    "Binarizada Invertida"),
    (mascara,    "Mascara (bitwise_and)"),
]
for ax, (img, titulo) in zip(axes.flat, pares):
    ax.imshow(img, cmap="gray")
    ax.set_title(titulo, fontsize=12, fontweight="bold")
    ax.axis("off")

plt.suptitle(f"Binarizacao Avancada — Limiar {LIMIAR_AV} | Blur 7x7", fontsize=14)
plt.tight_layout()
plt.savefig("resultado_avancado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura salva: resultado_avancado.png")

## 💾 7. Salvar Imagens Geradas

In [ ]:
os.makedirs("assets", exist_ok=True)

cv2.imwrite("assets/imagem_cinza.jpg",   img_cinza)
cv2.imwrite("assets/imagem_binaria.jpg", img_bin)
print("Salvo: assets/imagem_cinza.jpg")
print("Salvo: assets/imagem_binaria.jpg")

# Download automático no Google Colab
try:
    from google.colab import files
    for arq in ["assets/imagem_cinza.jpg", "assets/imagem_binaria.jpg",
                "resultado_binarizacao.png", "resultado_avancado.png"]:
        files.download(arq)
        print(f"Download: {arq}")
except ImportError:
    print("Ambiente local: arquivos salvos nas pastas acima.")